# 03 — RAG Pipeline with Citations

This notebook covers milestone point **7**: a working Retrieval-Augmented Generation pipeline that answers maintenance questions using retrieved motor/VFD documentation and returns explicit sources.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import VECTOR_DIR, COLLECTION_NAME

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in a .env file before running this notebook.')


## 1. Load the existing vector database

No PDF ingestion or embedding should happen here. That work was already completed in notebook 02 and cached in Chroma.

In [2]:
from factory_floor.vectorstore import get_embeddings, load_vectorstore
from factory_floor.rag import build_retriever

embeddings = get_embeddings()
vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=embeddings)
retriever = build_retriever(vectorstore, k=5)
print('Persistent retriever loaded.')


Persistent retriever loaded.


## 2. Define the maintenance RAG prompt

The model is explicitly told to use only retrieved context, admit when evidence is insufficient, and reference the source labels supplied in the context.

In [3]:
from factory_floor.rag import get_llm, PROMPT

llm = get_llm()
prompt = PROMPT


## 3. Build a deterministic citation layer

We label every retrieved chunk before it reaches the LLM and also append a source list programmatically. This makes the provenance visible even if the wording of the generated answer changes.

In [4]:
from factory_floor.rag import format_context, source_list


## 4. End-to-end RAG function

In [5]:
from factory_floor.rag import ask


def ask_factory_floor(question: str):
    return ask(question, retriever, llm)


## 5. Run the first end-to-end troubleshooting query

In [6]:
result = ask_factory_floor(
    'An industrial motor is overheating and has developed excessive vibration. What checks are supported by the manuals?'
)

print(result['answer'])
print('\nSOURCES USED')
print(result['sources'])

For an industrial motor overheating and exhibiting excessive vibration, the manuals support the following checks:

1. Overheating Checks:
   - Verify ambient temperature is within defined limits.
   - Check load conditions and duty cycle configuration.
   - Confirm cooling system functionality.
   - Inspect motor load and reduce if necessary.
   - Check wiring and connection of the motor temperature sensor (KTY84 or PT1000).
   - Verify motor overtemperature parameters such as thermal time constant (p0611) and fault threshold (p0605).
   - Check supply voltage parameterization (p0210) and line voltage.
   - Review torque limits (r1538, r1539) and current limits (p0640, r0067, r0289).
   - Ensure motor data is correctly parameterized and perform motor identification if needed [SOURCE 1, 3].

2. Vibration and Mechanical Checks:
   - Ensure all fixing bolts/screws for mechanical and electrical connections are securely tightened.
   - Confirm all potential, grounding, and shield connection

## 6. Test a VFD-oriented question

In [7]:
result = ask_factory_floor(
    'A SINAMICS G120 drive is repeatedly tripping. According to the retrieved documentation, what information should a technician inspect before deciding on a cause?'
)
print(result['answer'])
print('\nSOURCES USED')
print(result['sources'])

Before deciding on a cause for repeated tripping of a SINAMICS G120 drive, a technician should inspect the following information from the fault and alarm documentation:

1. Check for specific fault codes displayed on the drive, such as:
   - Internal communication faults (e.g., DRIVE-CLiQ wiring issues) [SOURCE 1].
   - Infeed faults indicating problems with line supply, filters, reactors, or fuses [SOURCE 1].
   - Braking module faults or overloads [SOURCE 1].
   - DC link voltage faults, including overvoltage or undervoltage conditions, which may be caused by motor regeneration, line supply voltage issues, or DC link voltage controller settings [SOURCE 2, SOURCE 3].
   - Motor overcurrent faults indicating motor overload or incorrect current limit settings [SOURCE 5].

2. Verify the environment and hardware conditions:
   - Check the air intake and fan of the Control Unit to ensure proper cooling [SOURCE 4].
   - Inspect wiring and connections for EMC compliance and integrity [SOURCE

## 7. Test an out-of-corpus question

A good RAG system should not confidently answer questions that are unsupported by its corpus.

In [8]:
result = ask_factory_floor('How do I repair a hydraulic excavator boom cylinder?')
print(result['answer'])
print('\nSOURCES USED')
print(result['sources'])

The retrieved documentation does not provide specific instructions for repairing a hydraulic excavator boom cylinder. The sources focus on maintenance and repair procedures for Siemens electric motors (SIMOTICS series), including general repair guidelines, sealing measures, screw lock washers, and reassembly instructions, but do not cover hydraulic cylinder repair.

Therefore, the documentation retrieved is insufficient to guide the repair of a hydraulic excavator boom cylinder. It is recommended to consult the hydraulic excavator or boom cylinder manufacturer's service manual or technical support for detailed repair procedures.

SOURCES USED
- [SOURCE 1] Siemens_SIMOTICS_SD_1LE7_Operating_Instructions.pdf — page 31
- [SOURCE 2] Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf — page 116
- [SOURCE 3] Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf — page 106
- [SOURCE 4] Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf — page 114
- [SOURCE 5] Siemens_SIMOTICS_GP_1LE1_Operatin

## Milestone checkpoint

We now have:

`question → semantic retrieval → technical context → LLM → grounded answer + citations`

This is the core RAG that the Streamlit interface will expose.